# RiskOptima Dislocation Model

This notebook demonstrates the additive `riskoptima.dislocation` module. Equities use quality + valuation + dislocation analytics; funds and pooled vehicles use price/regime dislocation without a fabricated corporate DCF.


### Execution environment

Run from the repository root with this branch installed (`python -m pip install -e .`). The live cells use the existing `yfinance` dependency and require internet access. One unavailable Yahoo ticker is returned as an error row rather than aborting the universe scan.


In [ ]:
from IPython.display import Markdown, display
import pandas as pd

from riskoptima.dislocation import (
    DislocationConfig,
    analyze_symbol,
    generate_markdown_report,
    scan_universe,
)

pd.set_option("display.max_columns", 30)


## Input universe

The symbols below are passed to Yahoo Finance exactly as supplied.


In [ ]:
INSTRUMENTS = [
    ("IE00B50W2R13.IR", "Global Bond Index Fund"),
    ("0P0000TKZI.L", "LifeStrategy 40% Equity Fund"),
    ("0P0000TKZO.L", "LifeStrategy 100% Equity Fund"),
    ("0P0000TKZM.L", "LifeStrategy 80% Equity Fund"),
    ("0P00017UWT.L", "Target Retirement 2045 Fund"),
    ("0P0000UGLG.L", "ESG Developed World All Cap Equity Index Fund"),
    ("0P0000UGLE.L", "ESG Developed Europe Index Fund"),
    ("0P0000KM22.L", "Emerging Markets Stock Index Fund"),
    ("0P0000KM20.L", "Pacific ex-Japan Stock Index Fund"),
    ("0P00018XAP.L", "FTSE 100 Index Unit Trust"),
    ("0P0000KSPA.L", "U.S. Equity Index Fund"),
]

pd.DataFrame(INSTRUMENTS, columns=["symbol", "name"])


## Run and rank all instruments


In [ ]:
config = DislocationConfig(history_period="5y", min_observations=126, use_markov_regime=True)
summary, reports = scan_universe(INSTRUMENTS, config=config, continue_on_error=True)
summary


## Generate the full report for every successful ticker

The default prose is generated from the measured state and is reproducible. `generate_markdown_report` also accepts a `narrative_generator` callback, so an LLM can be injected by an application without adding an LLM SDK dependency to RiskOptima.


In [ ]:
generated_reports = {report.symbol: generate_markdown_report(report) for report in reports}

for symbol, report_markdown in generated_reports.items():
    display(Markdown(report_markdown))
    display(Markdown("---"))


## Single-ticker equity example

For a genuine stock ticker, Yahoo fundamentals automatically activate the quality/valuation/FCFE path.


In [ ]:
# Example (uncomment when running live):
# equity_report = analyze_symbol("MSFT", config=config)
# display(Markdown(generate_markdown_report(equity_report)))


## Optional LLM narrative adapter

Applications can inject any callable accepting the report context and returning text. This keeps the quantitative model deterministic while allowing the final narrative to be genuinely generative.

```python
def my_generator(context: dict) -> str:
    # Call your preferred model/service here.
    return "..."

report = analyze_symbol("MSFT", narrative_generator=my_generator)
```
